In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


from masim_analysis import analysis, utils
from masim_analysis.configure import CountryParams


from ruamel.yaml import YAML


yaml = YAML()

country = CountryParams.load("moz")
country_code = "moz"


output = Path("output") / "moz"

In [ ]:
ave_cases = pd.read_csv(output / "validation" / "ave_cases.csv", index_col=0)
ave_pfpr = pd.read_csv(output / "validation" / "ave_prevalence_2_to_10.csv", index_col=0)
ave_population = pd.read_csv(output / "validation" / "ave_population.csv", index_col=0)

In [ ]:
ave_pfpr.drop(columns=["pfpr2to10"], inplace=True)

In [ ]:
cases_2023 = ave_cases.loc[ave_cases["monthlydataid"].between(144, 155)].copy()
pfpr_2023 = ave_pfpr.loc[ave_pfpr["monthlydataid"].between(144, 155)].copy()
cases_2023.drop(columns=["clinicalepisodes"], inplace=True)
population_2023 = ave_population.loc[ave_population["monthlydataid"].between(144, 155)].copy()
population_2023.drop(columns=["population"], inplace=True)

In [ ]:
cases_2023_monthly = cases_2023.groupby(["monthlydataid", "locationid"]).sum()
pfpr_2023_monthly = pfpr_2023.groupby(["monthlydataid", "locationid"]).mean()
population_2023_monthly = population_2023.groupby(["monthlydataid", "locationid"]).mean()

In [ ]:
cases_2023_mean = cases_2023_monthly.groupby("locationid").sum()
cases_2023_mean["cases"] = cases_2023_mean.mean(axis=1)

In [ ]:
pfpr_2023_mean = pfpr_2023_monthly.groupby("locationid").mean()
pfpr_2023_mean["pfpr2to10"] = pfpr_2023_mean.mean(axis=1)

In [ ]:
population_2023_mean = population_2023_monthly.groupby("locationid").mean()
population_2023_mean["population"] = population_2023_mean.mean(axis=1)

In [ ]:
print(f"Total cases in 2023: {cases_2023_mean[['cases']].sum().values[0] * 4:,}")

In [ ]:
district_raster, meta = utils.read_raster(Path("data") / country_code / "moz_districts.asc")
district_mapping = pd.read_csv("data/moz/moz_mapping.csv", index_col=0)

In [ ]:
pixel_ids = district_raster[~np.isnan(district_raster)].flatten()

In [ ]:
pixel_mappings = pd.DataFrame({"district": pixel_ids}, index=pd.Index(np.arange(len(pixel_ids)), name="locationid"))

In [ ]:
sim_results = cases_2023_mean["cases"].copy().to_frame()
sim_results = sim_results.merge(pfpr_2023_mean["pfpr2to10"].copy(), left_index=True, right_index=True, how="left")
sim_results = sim_results.merge(
    population_2023_mean["population"].copy(), left_index=True, right_index=True, how="left"
)
sim_results = sim_results.merge(pixel_mappings, left_index=True, right_index=True, how="left")
sim_results

In [ ]:
sim_results_districts = sim_results.groupby("district")[["cases", "population"]].sum()
sim_results_districts.rename(columns={"cases": "cases_sim", "population": "pop_sim"}, inplace=True)
sim_results_districts["pfpr_sim"] = sim_results.groupby("district")["pfpr2to10"].mean().div(100)
sim_results_districts

In [ ]:
pfpr = utils.read_raster(Path("data") / country_code / "moz_pfpr2to10.asc")[0]
pfpr_flat = pfpr[~np.isnan(district_raster)].flatten()

pop = utils.read_raster(Path("data") / country_code / "moz_population.asc")[0]
pop_flat = pop[~np.isnan(district_raster)].flatten()

In [ ]:
obs = pd.DataFrame(
    {"district": pixel_ids, "pfpr_obs": pfpr_flat}, index=pd.Index(np.arange(len(pixel_ids)), name="locationid")
)
obs["pop_obs"] = pop_flat

In [ ]:
obs

In [ ]:
obs_districts = obs.groupby("district")[["pfpr_obs"]].mean()
obs_districts["pop_obs"] = obs.groupby("district")["pop_obs"].sum()
obs_districts

In [ ]:
pop_comparison = (
    obs_districts["pop_obs"]
    .to_frame()
    .merge(sim_results_districts["pop_sim"] * 4, left_index=True, right_index=True, how="left")
)
pop_comparison["diff"] = pop_comparison["pop_sim"] - pop_comparison["pop_obs"]
pop_comparison = pop_comparison.merge(district_mapping, left_index=True, right_index=True, how="left")
pop_comparison.loc["sum"] = pop_comparison.sum(numeric_only=True)
pop_comparison.to_csv("moz_population_comparison.csv")

In [ ]:
comparison = obs["pfpr_obs"]  # .merge(sim_results_districts, left_index=True, right_index=True, how="left")
comparison = comparison.to_frame().merge(
    pfpr_2023_mean["pfpr2to10"].div(100), left_index=True, right_index=True, how="left"
)
comparison.rename(columns={"pfpr2to10": "pfpr_sim"}, inplace=True)

comparison = comparison.merge(obs["district"], left_index=True, right_index=True, how="left")
comparison_out = (
    comparison.groupby("district").mean().merge(district_mapping, left_index=True, right_index=True, how="left")
)
comparison_out["difference"] = comparison_out["pfpr_obs"] - comparison_out["pfpr_sim"]
comparison_out.to_csv("moz_pfpr_comparison.csv")

In [ ]:
comparison_out

In [ ]:
pfpr_sim = np.nan * np.ones(comparison.index[-1] + 1)
pfpr_sim[comparison.index.to_numpy()] = comparison["pfpr_sim"].to_numpy()
pfpr_sim_out = (np.nan * np.ones_like(pfpr)).flatten()
pfpr_sim_out[~np.isnan(pfpr.flatten())] = pfpr_sim
pfpr_sim_out = pfpr_sim_out.reshape(pfpr.shape)

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(10, 5))
im0 = ax[0].imshow(pfpr, vmin=0, vmax=0.5, cmap="coolwarm")
ax[0].set_title("Observed PfPR 2 to 10")
fig.colorbar(im0, ax=ax[0])
im1 = ax[1].imshow(pfpr_sim_out, vmin=0, vmax=0.5, cmap="coolwarm")
ax[1].set_title("Simulated PfPR 2 to 10")
fig.colorbar(im1, ax=ax[1])
im2 = ax[2].imshow(pfpr_sim_out - pfpr, cmap="bwr")
ax[2].set_title("Difference")
fig.colorbar(im2, ax=ax[2])
fig.savefig("moz_pfpr_comparison.png", dpi=300)
plt.show()

In [ ]:
moz_cases = pd.read_csv(Path("data") / country_code / "Cases 2023 2024.csv")

In [ ]:
# moz_2023 = moz_cases.loc[moz_cases["Ano"] == 2023].copy().groupby("district").sum()
# moz_2023.drop(columns={"Ano", "Provincia"}, inplace=True)
moz_2023 = moz_cases

In [ ]:
moz_2023

In [ ]:
sim_results_districts

In [ ]:
moz_2023

In [ ]:
sim_results_districts.loc["sum"] = sim_results_districts.sum()

In [ ]:
case_comparison = moz_2023["Adjusted Anual Cases"].copy().to_frame()
case_comparison = case_comparison.merge(
    sim_results_districts[["cases_sim", "pop_sim"]] * 4, left_index=True, right_index=True, how="left"
)
case_comparison.rename(columns={"Adjusted Anual Cases": "observed_cases", "cases": "simulated_cases"}, inplace=True)
# case_comparison.loc["sum"] = case_comparison.sum()
# case_comparison["diff"] = case_comparison["simulated_cases"] - case_comparison["observed_cases"]
case_comparison = case_comparison.merge(district_mapping, left_index=True, right_index=True, how="left")
case_comparison["difference"] = case_comparison["observed_cases"] - case_comparison["cases_sim"]

case_comparison["sim_percentage"] = 100 * case_comparison["cases_sim"] / case_comparison["cases_sim"].sum()
case_comparison["obs_percentage"] = 100 * case_comparison["observed_cases"] / case_comparison["observed_cases"].sum()
case_comparison["diff_percentage"] = case_comparison["obs_percentage"] - case_comparison["sim_percentage"]

case_comparison.loc["sum"] = case_comparison.sum(numeric_only=True)

case_comparison.to_csv("moz_case_comparison.csv")

In [ ]:
case_comparison

In [ ]:
case_comparison_percentage = case_comparison.copy()

In [ ]:
proportion_comparison = case_comparison.copy()
proportion_comparison["observed_proportion"] = (
    proportion_comparison["observed_cases"] / proportion_comparison["observed_cases"].sum()
)
proportion_comparison["simulated_proportion"] = (
    proportion_comparison["cases_sim"] / proportion_comparison["cases_sim"].sum()
)
proportion_comparison.drop(columns={"observed_cases", "cases_sim"}, inplace=True)
proportion_comparison

In [ ]:
proportion_comparison = proportion_comparison.merge(district_mapping, left_index=True, right_index=True, how="left")

In [ ]:
proportion_comparison.to_csv("moz_case_proportion_comparison.csv")

In [ ]:
case_comparison = case_comparison.merge(district_mapping, left_index=True, right_index=True, how="left")

In [ ]:
case_comparison.to_csv("moz_case_comparison.csv")

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
proportion_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# scatter points
x = proportion_comparison["observed_proportion"]
y = proportion_comparison["simulated_proportion"]
ax.scatter(x, y, s=50, color="C0")

# annotate each point with the district name (fall back to index if no name)
for idx in x.index:
    xi, yi = x.loc[idx], y.loc[idx]
    label = district_mapping.loc[idx, "DISTRICT"] if idx in district_mapping.index else str(idx)
    ax.annotate(label, (xi, yi), xytext=(5, 5), textcoords="offset points", fontsize=9, ha="left", va="bottom")
ax.plot(np.linspace(0, 1), np.linspace(0, 1), "r--")
ax.set_xlim((0, 0.3))
ax.set_ylim((0, 0.25))
ax.set_xlabel("Observed")
ax.set_ylabel("Simulated")
ax.set_title("Observed vs. Simulated Annual Case Proportions")
fig.savefig(Path("images") / "moz" / "proportions.png")

In [ ]:
np.arange(0, 1, 1000)

---

In [ ]:
genotypes = analysis.get_table(output / "validation" / "moz_validation_scaled_0.37_monthly_data_0.db", "genotype")
genome_data = analysis.get_table(
    output / "validation" / "moz_validation_scaled_0.37_monthly_data_0.db", "monthlygenomedata"
)

In [ ]:
genome_data.loc[genome_data["monthlydataid"] == 144]

In [ ]:
genome_data_2022 = genome_data.copy().loc[genome_data["monthlydataid"].between(144, 155)]

In [ ]:
genome_data_2022

In [ ]:
genome_data_2022 = (
    genome_data_2022[
        ["genomeid", "occurrences", "clinicaloccurrences", "occurrences0to5", "occurrences2to10", "weightedoccurrences"]
    ]
    .groupby("genomeid")
    .sum()
)
genome_data_2022

In [ ]:
genotypes.loc[genotypes["id"] == 36]

In [ ]:
genome_data_2022 = pd.DataFrame()

for file in output.glob("validation/*.db"):
    in_data = analysis.get_table(file, "monthlygenomedata")
    in_data_2022 = in_data.loc[in_data["monthlydataid"].between(144, 155)]
    in_data_2022_grouped = (
        in_data_2022[
            [
                "genomeid",
                "occurrences",
                "clinicaloccurrences",
                "occurrences0to5",
                "occurrences2to10",
                "weightedoccurrences",
            ]
        ]
        .groupby("genomeid")
        .sum()
    )
    genome_data_2022 = pd.concat([genome_data_2022, in_data_2022_grouped])

# genome_data_2022 = genome_data_2022.groupby("genomeid").mean()
genome_data_2022.to_csv(output / "validation" / "genome_data_2022_sum.csv")

In [ ]:
genome_data_2022 = pd.read_csv(output / "validation" / "genome_data_2022_mean.csv", index_col=0)
genome_data_2022 = genome_data_2022.groupby("genomeid").mean() * 4
genome_data_2022 = genome_data_2022.merge(genotypes, left_index=True, right_on="id")
genome_data_2022 = genome_data_2022.drop(columns=["id"])
genome_data_2022

In [ ]:
genome_records = pd.read_csv("data/moz/moz genome data.csv")
genome_records = genome_records[
    ["gene", "locus", "allele", "n_total_samples_collected_day_0", "n_carriers_identified_day_0", "frequency"]
]
genome_records

### Reported Allele Frequencies in 2022

In [ ]:
genome_records = genome_records.groupby(["gene", "locus", "allele"]).sum()
genome_records["frequency"] = (
    genome_records["n_carriers_identified_day_0"] / genome_records["n_total_samples_collected_day_0"]
)
genome_records

### Simulation Allele Frequencies in 2022

In [ ]:
genome_data_2022["frequency"] = genome_data_2022["clinicaloccurrences"] / genome_data_2022["occurrences"]
genome_data_2022

In [ ]:
genome_data_2022["occurrences"].sum()

TNY--R1: pfcrt + pfmdr1

TYY--R1: pfcrt + pfmdr1

In [ ]:
from masim_analysis import calibrate

In [ ]:
models_map = calibrate.load_beta_model("data/moz/models_map.json")

In [ ]:
population_raster, meta = utils.read_raster("data/moz/moz_population.asc")
access_rate_raster = utils.read_raster("data/moz/moz_treatmentseeking.asc")[0]
prevalence_raster = utils.read_raster("data/moz/moz_pfpr210.asc")[0]

In [ ]:
beta_map = calibrate.create_beta_map(models_map, population_raster, access_rate_raster, prevalence_raster)

In [ ]:
utils.write_raster(beta_map, "output/moz/moz_beta_map.asc", meta["xllcorner"], meta["yllcorner"])

In [ ]:
beta_raster = utils.read_raster("data/moz/moz_beta_scaled_0.37.asc")[0]

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
from matplotlib.pyplot import clim


fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(beta_map * 0.37, cmap="viridis")
im.set_clim(0, 1.0)
cbar = fig.colorbar(im, ax=ax, label="Beta Value")
ax.set_title("Mozambique Scaled Beta Raster")
fig.savefig("output/moz/moz_beta_map_scaled.png")
plt.show()

---

In [ ]:
analysis.get_all_tables("moz_validation_scaled_0.37_mut_monthly_data_99.db")

In [ ]:
genotype_data = analysis.get_table("moz_validation_scaled_0.37_monthly_data_1.db", "monthlygenomedata")

In [ ]:
genotype_data

In [ ]:
genotype_data["genomeid"].unique()

In [ ]:
monthlysitedata = analysis.get_table("moz_validation_scaled_0.37_monthly_data_1.db", "monthlysitedata")

In [ ]:
monthlysitedata